# 🏗️ Notebook 1: Netflix — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/netflix
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A global video streaming service. Think Netflix: millions of users, petabytes of video,
low startup latency worldwide, personalized recommendations.

### Functional requirements (the happy path)
- Upload & publish a video (admin side) — transcoded into multiple resolutions.
- Browse and search the catalog.
- **Stream** a video with good startup time and minimal rebuffering.
- Personalized **recommendations** ("Because you watched…").
- Track **playback position** so users resume where they left off.

### Non-functional
- **Read-heavy**: 99% of traffic is *watching*, not *uploading*. Optimize for reads.
- **Low startup**: <2s from click to first frame.
- **High availability**: 99.99%+ for playback.
- **Global**: users in Asia should not pull bytes from US-East.


## Back-of-envelope

- 200M users. Peak concurrent viewers: ~15% = 30M.
- Average bitrate: 3 Mbps (SD/HD mix).
- Peak egress: 30M × 3 Mbps ≈ **90 Tbps**. Single data center can't serve this → **CDN is mandatory**.
- Catalog: 20k titles × avg 5 GB × 6 renditions ≈ **600 TB** of mezzanine + renditions.


## High-level architecture

```
  [User]
    │ https
    ▼
  ┌────────────────────┐
  │   Edge CDN (POPs)  │◀── 99% of bytes served here
  └────────┬───────────┘
           │ cache miss / metadata
           ▼
  ┌────────────────────┐   ┌─────────────────┐
  │   API Gateway      │──▶│  Auth, Rate lim │
  └────────┬───────────┘   └─────────────────┘
           │
   ┌───────┼───────┬───────────────┬──────────────┐
   ▼       ▼       ▼               ▼              ▼
 Catalog  User   Playback       Recommendation  Watch-history
 Service  Svc    Svc (manifests)   Svc           Svc
   │       │       │                │              │
   ▼       ▼       ▼                ▼              ▼
 MySQL  MySQL   Object Store    ML feature   Cassandra
                (S3)            store + models
                   ▲
                   │ write after encoding
  ┌────────────────┴──────────────────┐
  │     Encoding / transcoding        │  (batch jobs, ffmpeg workers)
  │  mezzanine → HLS/DASH × bitrates  │
  └───────────────────────────────────┘
```

- **CDN**: primary *bytes* delivery. Netflix has its own (OpenConnect); a design could use Cloudflare/Akamai.
- **Manifest**: a small playlist (HLS `.m3u8` or DASH MPD) telling the player which chunks to fetch at which bitrate.
- **ABR (Adaptive Bit Rate)**: player picks quality based on measured bandwidth.
- **Recommendations** are precomputed offline; served as a lookup.
